# German Legal QA – LoRA Fine-Tuning (Colab T4)

Trains Llama-3.2-3B-Instruct with QLoRA on German legal QA data.

**BEFORE RUNNING:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Install exact versions that work together
!pip install -q \
    torch==2.4.0 \
    transformers==4.44.2 \
    datasets==2.21.0 \
    accelerate==0.33.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    trl==0.9.6 \
    sentencepiece \
    huggingface_hub

In [ ]:
import torch
assert torch.cuda.is_available(), "NO GPU! Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

In [ ]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN_HERE")  # Get from https://huggingface.co/settings/tokens

## 1. Load Data

In [ ]:
import requests, json
from datasets import Dataset

base_url = "https://raw.githubusercontent.com/trusthlt/eacl24-german-legal-questions/main/data"
all_records = []
for fname in ["GerLayQA.json", "stgb_QA.json", "zpo_QA.json"]:
    resp = requests.get(f"{base_url}/{fname}")
    if resp.status_code == 200:
        data = resp.json()
        records = data if isinstance(data, list) else [data]
        for item in records:
            q = item.get("Question_text", "") or item.get("question", "")
            a = item.get("Answer_text", "") or item.get("answer", "")
            if q and a and len(q) > 10 and len(a) > 10:
                all_records.append({"instruction": q, "output": a})

dataset = Dataset.from_list(all_records)
print(f"Dataset: {len(dataset)} examples")
print(f"Sample Q: {dataset[0]['instruction'][:100]}...")

## 2. Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

print(f"Model device: {model.device}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Apply LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Prepare Training Data

In [ ]:
SYSTEM_MSG = (
    "Du bist ein deutschsprachiger Rechtsassistent. Beantworte rechtliche Fragen "
    "präzise und verständlich. Zitiere relevante Paragraphen wenn möglich. "
    "Dies ist keine Rechtsberatung."
)

def format_example(example):
    """Format a single example into a full text string for training."""
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return text

# Pre-tokenize the dataset (avoids formatting_func issues)
def tokenize_function(examples):
    texts = []
    for inst, out in zip(examples["instruction"], examples["output"]):
        messages = [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": inst},
            {"role": "assistant", "content": out},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors=None,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=dataset.column_names,
    desc="Tokenizing",
)
print(f"Tokenized: {len(tokenized_dataset)} examples")
print(f"Sample length: {len(tokenized_dataset[0]['input_ids'])} tokens")

## 4. Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./legal-qa-3b-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_steps=50,
    weight_decay=0.01,
    fp16=True,
    logging_steps=25,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    max_steps=1500,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print(f"Training steps: {training_args.max_steps}")
print(f"Batch size: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print("\nStarting training...")
trainer.train()

## 5. Save & Download

In [ ]:
# Save adapter
adapter_path = "./legal-qa-3b-lora/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

import json
info = {
    "model_id": MODEL_ID,
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 2,
    "max_steps": 1500,
    "training_samples": len(dataset),
}
with open(f"{adapter_path}/training_info.json", "w") as f:
    json.dump(info, f, indent=2)

print(f"Adapter saved to {adapter_path}")
!du -sh {adapter_path}

In [ ]:
# Copy to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/german-legal-qa-adapter
!cp -r {adapter_path}/* /content/drive/MyDrive/german-legal-qa-adapter/
print("Copied to Google Drive!")

## 6. Test

In [ ]:
model.eval()
question = "Was besagt §823 BGB?"
messages = [
    {"role": "system", "content": SYSTEM_MSG},
    {"role": "user", "content": question},
]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Q: {question}\n\nA: {response}")

## 7. (Optional) Push to HuggingFace

Uncomment and set your username:

In [ ]:
# model.push_to_hub("YOUR_USERNAME/german-legal-qa-3b-lora")
# tokenizer.push_to_hub("YOUR_USERNAME/german-legal-qa-3b-lora")